In [2]:
import numpy as np
import tensorflow as tf
import tensorflow_datasets as tfds

In [3]:
mnist_dataset, mnist_info = tfds.load(name='mnist', with_info=True, as_supervised=True)

Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Extraction completed...: 0 file [00:00, ? file/s]

Generating splits...:   0%|          | 0/2 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /Users/istamjon/tensorflow_datasets/mnist/incomplete.1NEEXI_3.0.1/mnist-train.tfrecord-[0-9][0-9][0-…

Generating test examples...: 0 examples [00:00, ? examples/s]

Shuffling /Users/istamjon/tensorflow_datasets/mnist/incomplete.1NEEXI_3.0.1/mnist-test.tfrecord-[0-9][0-9][0-9…

Dataset mnist downloaded and prepared to /Users/istamjon/tensorflow_datasets/mnist/3.0.1. Subsequent calls will reuse this data.


### Data

In [7]:
mnist_train, mnist_test = mnist_dataset['train'], mnist_dataset['test']

num_validation_samples = 0.1 * mnist_info.splits['train'].num_examples
num_validation_samples = tf.cast(num_validation_samples, tf.int64)

num_test_samples = mnist_info.splits['test'].num_examples
num_test_samples = tf.cast(num_test_samples, tf.int64)

def scale(image, label):
     image = tf.cast(image, tf.float32)
     image /= 255.
     return image, label

scaled_train_validation_data = mnist_train.map(scale)
test_data = mnist_test.map(scale)

buffer_size = 10000
shuffled_train_validation_data = scaled_train_validation_data.shuffle(buffer_size)

validation_data = shuffled_train_validation_data.take(num_validation_samples)
train_data = shuffled_train_validation_data.skip(num_validation_samples)

batch_size = 100
train_data = train_data.batch(batch_size)
validation_data = validation_data.batch(num_validation_samples)
test_data = test_data.batch(num_test_samples)

validation_inputs, validation_targets = next(iter(validation_data))

I0000 00:00:1794917467.480790 12081570 tf_record_dataset_op.cc:398] The default buffer size is 262144, which is overridden by the user specified `buffer_size` of 8388608
W0000 00:00:1794917467.634274 12081574 cache_dataset_ops.cc:956] The calling iterator did not fully read the dataset being cached. In order to avoid unexpected truncation of the dataset, the partially cached contents of the dataset will be discarded. This can happen if you have an input pipeline similar to `dataset.cache().take(k).repeat()`, or if downstream operations drop elements (e.g. `batch(drop_remainder=True)`). You should use `dataset.take(k).cache().repeat()` instead, or ensure the dataset size is a multiple of the batch size before caching. Another common workaround is to place the `.cache()` operation after the operation that drops elements (like `.batch(...)`), if caching the transformed data is acceptable.


### Model

In [48]:
input_size = 784
output_size = 10
hidden_layer_size = 200

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(28, 28, 1)),
    tf.keras.layers.Flatten(),
    
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    tf.keras.layers.Dense(hidden_layer_size, activation='relu'),
    
    tf.keras.layers.Dense(output_size, activation='softmax')
])

In [49]:
model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])

In [50]:
num_epochs = 5
model.fit(train_data, epochs=num_epochs, validation_data=(validation_inputs, validation_targets), verbose=2)

Epoch 1/5
540/540 - 1s - 2ms/step - accuracy: 0.9216 - loss: 0.2646 - val_accuracy: 0.9610 - val_loss: 0.1226
Epoch 2/5
540/540 - 1s - 1ms/step - accuracy: 0.9702 - loss: 0.0991 - val_accuracy: 0.9758 - val_loss: 0.0764
Epoch 3/5
540/540 - 1s - 1ms/step - accuracy: 0.9787 - loss: 0.0691 - val_accuracy: 0.9827 - val_loss: 0.0541
Epoch 4/5
540/540 - 1s - 1ms/step - accuracy: 0.9839 - loss: 0.0499 - val_accuracy: 0.9830 - val_loss: 0.0498
Epoch 5/5
540/540 - 1s - 1ms/step - accuracy: 0.9866 - loss: 0.0420 - val_accuracy: 0.9868 - val_loss: 0.0440


### Test

In [51]:
test_loss, test_accuracy = model.evaluate(test_data)

print('Test loss: {0: .2f}, test accuracy: {1:.2f}%')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 125ms/step - accuracy: 0.9788 - loss: 0.0671
<_BatchDataset element_spec=(TensorSpec(shape=(None, 28, 28, 1), dtype=tf.float32, name=None), TensorSpec(shape=(None,), dtype=tf.int64, name=None))> 0.06707629561424255
